# Lab 5: Function Tool Agent — Custom Business Logic
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Extend your agent with custom business functions. Unlike Code Interpreter (which runs arbitrary code), Function Tools let you expose **specific, controlled operations** to the agent. You’ll learn:
- Defining function schemas (JSON Schema format)
- Registering custom functions as agent tools
- The **function call loop** — how the agent requests function execution
- Handling function calls client-side and returning results

## Architecture
```
User Query → Agent decides to call function → Returns function_call → 
Client executes function locally → Sends result back → Agent generates final response
```

> 💡 **Key Difference:** Code Interpreter runs code *server-side* in a sandbox. Function Tools run *client-side* — your code, your data, your security controls.

## Step 1: Import Dependencies

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import (
    get_clients, MODEL, print_header, print_step, ask_with_functions,
)
from utils.business_functions import get_claim_status, calculate_fraud_risk
from azure.ai.projects.models import PromptAgentDefinition

project_client, openai_client = get_clients()
print("\u2705 Clients created")

## Step 2: Define Function Schemas

Each function tool needs a **JSON Schema** definition that tells the agent:
- **What the function does** (description)
- **What parameters it accepts** (properties, types, enums)
- **Which parameters are required**

The agent uses these schemas to decide *when* to call a function and *how* to construct the arguments.

### Function 1: `get_claim_status`
Looks up a claim by ID from the CSV database. Simulates a CRM/database query.

### Function 2: `calculate_fraud_risk`
Calculates fraud risk score using a rules-based model. Simulates an ML fraud detection service.

In [ ]:
FUNCTION_MAP = {
    "get_claim_status": get_claim_status,
    "calculate_fraud_risk": calculate_fraud_risk,
}

FUNCTION_TOOLS = [
    {
        "type": "function",
        "name": "get_claim_status",
        "description": (
            "Look up the current status of an insurance claim by ID. "
            "Returns claim details including status, amount, adjuster, "
            "and fraud score."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "claim_id": {
                    "type": "string",
                    "description": "Claim ID in CLM-XXXX format (e.g., CLM-0042)"
                }
            },
            "required": ["claim_id"]
        }
    },
    {
        "type": "function",
        "name": "calculate_fraud_risk",
        "description": (
            "Calculate fraud risk score for a new insurance claim. "
            "Returns risk score (0-1), risk level, and recommendation."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "incident_type": {
                    "type": "string",
                    "description": "Type of incident",
                    "enum": [
                        "Auto Collision", "Property Damage",
                        "Medical Claim", "Theft",
                        "Natural Disaster", "Liability",
                        "Fire Damage"
                    ]
                },
                "claim_amount": {
                    "type": "number",
                    "description": "Dollar amount of the claim"
                },
                "region": {
                    "type": "string",
                    "description": "Geographic region",
                    "enum": ["North", "South", "East", "West", "Central"]
                },
                "days_since_policy_start": {
                    "type": "integer",
                    "description": "Days since policy was activated"
                }
            },
            "required": [
                "incident_type", "claim_amount",
                "region", "days_since_policy_start"
            ]
        }
    },
]

print("Registered function tools:")
for tool in FUNCTION_TOOLS:
    params = list(tool["parameters"]["properties"].keys())
    print(f"  \u2705 {tool['name']}({', '.join(params)})")

## Step 3: Create Agent with Function Tools

The agent receives the function schemas in its `tools` parameter. It will autonomously decide when to call these functions based on user queries.

In [ ]:
agent = project_client.agents.create_version(
    agent_name="smartclaims-functions",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims Operations Agent. Tools available: "
            "(1) get_claim_status - look up any claim by ID (CLM-XXXX). "
            "(2) calculate_fraud_risk - assess fraud risk for new claims. "
            "Always use tools when relevant. Present results clearly "
            "with key details highlighted."
        ),
        tools=FUNCTION_TOOLS,
    ),
)
print(f"Agent: {agent.name} v{agent.version}")

## Step 4: The Function Call Loop

When the agent decides to call a function, here's what happens:

```
1. Agent receives user query
2. Agent returns a `function_call` output (not text)
3. Client extracts function name + arguments
4. Client executes the function locally
5. Client sends the result back as `function_call_output`
6. Agent uses the result to generate its final text response
```

The `ask_with_functions()` helper from `utils/config.py` handles this loop automatically, including support for multiple consecutive function calls.

> ⚠️ **Important:** Each test uses a fresh conversation to avoid `call_id` conflicts between requests.

### Test 1: Claim Lookup
The agent should call `get_claim_status` with the claim ID.

In [ ]:
conv = openai_client.conversations.create()
answer = ask_with_functions(
    openai_client, agent, conv.id,
    "What is the status of claim CLM-0042?",
    FUNCTION_TOOLS, FUNCTION_MAP,
)
print(f"\n\ud83e\udd16 Agent: {answer}")

### Test 2: Fraud Risk Assessment
The agent should call `calculate_fraud_risk` with the provided parameters.

In [ ]:
conv = openai_client.conversations.create()
answer = ask_with_functions(
    openai_client, agent, conv.id,
    "Calculate fraud risk for a $85,000 Theft claim from the "
    "West region. Policy started 45 days ago.",
    FUNCTION_TOOLS, FUNCTION_MAP,
)
print(f"\n\ud83e\udd16 Agent: {answer}")

### Test 3: High-Risk Scenario
New policy + high amount + fire damage = likely high fraud risk.

In [ ]:
conv = openai_client.conversations.create()
answer = ask_with_functions(
    openai_client, agent, conv.id,
    "New Fire Damage claim: $350,000, Central region, policy "
    "only 20 days old. What is the fraud risk?",
    FUNCTION_TOOLS, FUNCTION_MAP,
)
print(f"\n\ud83e\udd16 Agent: {answer}")

### Test 4: Combined Query
The agent may need to call multiple functions — first look up the claim, then assess fraud risk based on the details.

In [ ]:
conv = openai_client.conversations.create()
answer = ask_with_functions(
    openai_client, agent, conv.id,
    "Look up CLM-0100. Based on the details, does the existing "
    "fraud score seem appropriate?",
    FUNCTION_TOOLS, FUNCTION_MAP,
)
print(f"\n\ud83e\udd16 Agent: {answer}")

## Step 5: Clean Up

In [ ]:
project_client.agents.delete(agent.name)
print("\u2705 Agent deleted")

## ✅ Lab 5 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|-----------------|
| Function schemas | JSON Schema defines function interface for the agent |
| Function call loop | Agent requests → Client executes → Client returns result |
| `FUNCTION_MAP` | Registry mapping function names to Python callables |
| `ask_with_functions()` | Utility that handles the full function call loop |
| Client-side execution | Functions run in YOUR environment, not a sandbox |

### Function Tools vs Code Interpreter
| Feature | Function Tools | Code Interpreter |
|---------|---------------|------------------|
| Execution location | Client-side | Server sandbox |
| Code written by | Developer (predefined) | Agent (generated) |
| Data access | Your databases/APIs | Uploaded files only |
| Security | Your controls | Sandboxed |
| Best for | Business logic, APIs | Ad-hoc analysis |

---
**Next →** [Lab 6: Multi-Tool Agent](lab6_multi_tool.ipynb) — Combine all tool types into one powerful agent